<a href="https://colab.research.google.com/github/Peeyusj/FashionMNIST/blob/main/FashionMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 201kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.77MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.21MB/s]


In [3]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size,shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [6]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [7]:
train(train_dataloader, model, loss_fn, optimizer)

loss: 2.299319  [   64/60000]
loss: 2.290174  [ 6464/60000]
loss: 2.275855  [12864/60000]
loss: 2.251163  [19264/60000]
loss: 2.237353  [25664/60000]
loss: 2.222064  [32064/60000]
loss: 2.207123  [38464/60000]
loss: 2.180368  [44864/60000]
loss: 2.173753  [51264/60000]
loss: 2.145301  [57664/60000]


In [8]:
def test(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.eval()
    with torch.no_grad():
      for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [9]:
for epoch in range(5):
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn, optimizer)

loss: 2.124138  [   64/60000]
loss: 2.112422  [ 6464/60000]
loss: 2.086467  [12864/60000]
loss: 2.071519  [19264/60000]
loss: 2.030206  [25664/60000]
loss: 2.043471  [32064/60000]
loss: 1.971065  [38464/60000]
loss: 1.952831  [44864/60000]
loss: 1.907955  [51264/60000]
loss: 1.837151  [57664/60000]
loss: 1.873509  [   64/10000]
loss: 1.867478  [ 6464/10000]
loss: 1.865016  [   64/60000]
loss: 1.875626  [ 6464/60000]
loss: 1.744420  [12864/60000]
loss: 1.809058  [19264/60000]
loss: 1.656105  [25664/60000]
loss: 1.688552  [32064/60000]
loss: 1.627027  [38464/60000]
loss: 1.573897  [44864/60000]
loss: 1.500541  [51264/60000]
loss: 1.558982  [57664/60000]
loss: 1.507861  [   64/10000]
loss: 1.537176  [ 6464/10000]
loss: 1.490460  [   64/60000]
loss: 1.422746  [ 6464/60000]
loss: 1.428883  [12864/60000]
loss: 1.400878  [19264/60000]
loss: 1.558183  [25664/60000]
loss: 1.291416  [32064/60000]
loss: 1.246344  [38464/60000]
loss: 1.413407  [44864/60000]
loss: 1.276684  [51264/60000]
loss: 1.21